In [1]:
import tensorflow as tf
import os, yaml, h5py

from msfm.utils import files
from deep_lss.utils import configuration
from deep_lss.models.base_model import BaseModel
from deep_lss.nets import NETWORKS

24-04-24 05:29:20   imports.py INF   Setting up healpy to run on 256 CPUs 


In [2]:
dir_model = "/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla"
checkpoint_dir = os.path.abspath(os.path.join(dir_model, "checkpoint"))
n_output = 5


with open(os.path.join(dir_model, "configs.yaml"), "r") as f:
    net_conf, dlss_conf, msfm_conf = list(yaml.load_all(f, Loader=yaml.FullLoader))
    
loss_func = net_conf["run"]["loss_func"]
n_side = msfm_conf["analysis"]["n_side"]
n_z_bins = 4

smoothing_kwargs = configuration.get_smoothing_kwargs(
    loss_func, msfm_conf, dlss_conf, net_conf, dir_base=dir_model
)

data_vec_pix, _, _, _ = files.load_pixel_file(msfm_conf)

network = NETWORKS[net_conf["network"]["name"]](
    out_features=n_output, smoothing_kwargs=smoothing_kwargs, **net_conf["network"]["kwargs"]
).get_layers()

model = BaseModel(
    network=network,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=net_conf["network"]["n_neighbors"],
    input_shape=(None, len(data_vec_pix), n_z_bins),
    checkpoint_dir=checkpoint_dir,
    # always load from a checkpoint
    restore_checkpoint=True,
    strategy=None,
)


24-04-24 05:29:24     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_512.h5 
24-04-24 05:29:24     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_512.h5 
24-04-24 05:29:24     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_512.h5 
Using the per channel smoothing repetitions [6 3 2 1]
Using the per channel smoothing scales sigma = [9.78 6.91 5.65 3.99] arcmin, fwhm = [23.03 16.28 13.29  9.4 ] arcmin
Successfully loaded sparse kernel indices and values from /pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/smoothing
Successfully created the sparse kernel tensor
24-04-24 05:29:26 regression_h INF   Using a dense regression head 
24-04-24 05:29:26 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the inpu

In [3]:
test_batch = tf.random.uniform((1, len(data_vec_pix), n_z_bins))

print(model(test_batch))

checkpoints = model.checkpoint_manager.checkpoints
print(checkpoints)

for checkpoint in checkpoints:
    model.checkpoint_manager.checkpoint.restore(checkpoint)
    print(model(test_batch))

tf.Tensor([[  0.4790032  -2.6545033 -18.894152  -23.254427    3.067468 ]], shape=(1, 5), dtype=float32)
['/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/checkpoint/ckpt-11', '/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/checkpoint/ckpt-12', '/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/checkpoint/ckpt-13', '/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/checkpoint/ckpt-14', '/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/checkpoint/ckpt-15', '/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/checkpoint/ckpt-16', '/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/checkpoint/ckpt-17', '/pscratch/sd/a/athomsen/run_files/v8/lensing/delta/2024-04-22_06-50-39_resnet_vanilla/checkpoint/ckpt-18', '/pscratch/sd/a/athomsen/run_fi

In [14]:
steps = [55_000, 60_000, 65_000, 70_000, 75_000]
for step in steps:
    with h5py.File(os.path.join(dir_model, f"preds_{step}_test.h5")) as f:
        print(f["fiducial/vali/pred"][0,:])

[-0.09383138 -0.06514484 -0.06345515  0.01005747  0.00400154]
[ 0.02261829  0.01571814  0.05417755  0.21897578 -0.05508681]
[ 0.0181499   0.00150289  0.00140993  0.29926035 -0.01648016]
[ 0.06762911  0.04131591  0.19044393 -0.02959165 -0.00187312]
[-0.10905204 -0.02537104  0.05859148  0.17214938 -0.07169797]
